In [9]:
import os
import cv2
import torch
import numpy as np
import xml.etree.ElementTree as ET
from xml.dom import minidom
from ultralytics import YOLO
import segmentation_models_pytorch as smp
from tqdm import tqdm
from datetime import datetime

# ===================== CONFIG =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_IMG_DIR = "../00_data/DIVA-HisDB/CB55/img-CB55/img/public-test/"
TEST_XML_DIR = "../00_data/DIVA-HisDB/CB55/PAGE-gt-CB55-TASK-2/TASK-2/public-test/"
OUT_XML_DIR = "./predictions_cb55/"

YOLO_WEIGHTS = "../runs/detect/yolo_CB55_5shots_experiment/weights/best.pt"
SEG_WEIGHTS = "../80_models/seg_segformer_yolo_CB55_5shots.pth"

RESIZE_H, RESIZE_W = 256, 256
BIN_THRESH = 0.1
# ==================================================

os.makedirs(OUT_XML_DIR, exist_ok=True)
print(f"[INFO] Device: {DEVICE}")

# ===================== MODELS =====================
print("[INFO] Loading YOLO...")
yolo = YOLO(YOLO_WEIGHTS)

print("[INFO] Loading segmentation model...")
seg_model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    decoder_dropout=0.2
).to(DEVICE)

seg_model.load_state_dict(torch.load(SEG_WEIGHTS, map_location=DEVICE))
seg_model.eval()
print("[INFO] Models ready")
# ==================================================

# ===================== HELPERS =====================
def segment_crop(crop_bgr):
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    h0, w0 = crop_rgb.shape[:2]

    crop_resized = cv2.resize(crop_rgb, (RESIZE_W, RESIZE_H))
    tensor = torch.from_numpy(crop_resized).permute(2,0,1).float() / 255.0
    tensor = tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.sigmoid(seg_model(tensor))[0,0].cpu().numpy()

    probs = cv2.resize(probs, (w0, h0))
    return (probs >= BIN_THRESH).astype(np.uint8) * 255


def mask_to_polygon(mask, ox, oy):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    return [(int(p[0][0] + ox), int(p[0][1] + oy)) for p in cnt]


def polygon_to_baseline(poly):
    poly = np.array(poly)
    y_max = poly[:,1].max()
    base = poly[poly[:,1] >= y_max - 2]
    base = base[np.argsort(base[:,0])]
    return [(int(x), int(y)) for x,y in base]


def create_page_xml(img_path, W, H):
    root = ET.Element("PcGts", {
        "xmlns": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15",
        "xmlns:xsi": "http://www.w3.org/2001/XMLSchema-instance",
        "xsi:schemaLocation":
            "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15 "
            "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15/pagecontent.xsd"
    })

    meta = ET.SubElement(root, "Metadata")
    ET.SubElement(meta, "Creator").text = "Arthur XML Generator"
    ET.SubElement(meta, "Created").text = datetime.now().isoformat()
    ET.SubElement(meta, "LastChange").text = datetime.now().isoformat()

    page = ET.SubElement(root, "Page", {
        "imageFilename": os.path.basename(img_path),
        "imageWidth": str(W),
        "imageHeight": str(H)
    })

    region = ET.SubElement(page, "TextRegion", {"id": "region_textline"})
    ET.SubElement(region, "Coords", {
        "points": f"0,0 {W},0 {W},{H} 0,{H}"
    })

    return root, region
# ==================================================

# ===================== PIPELINE =====================
img_files = sorted([
    f for f in os.listdir(TEST_IMG_DIR)
    if f.lower().endswith((".jpg", ".png"))
])

print(f"[INFO] Found {len(img_files)} images")

for img_name in tqdm(img_files, desc="Processing images"):
    img_path = os.path.join(TEST_IMG_DIR, img_name)
    image = cv2.imread(img_path)

    if image is None:
        print(f"[WARN] Failed to read {img_name}")
        continue

    H, W = image.shape[:2]

    results = yolo(image, verbose=False)[0]
    boxes = results.boxes.xyxy.cpu().numpy()

    # read original xml, extract main region coordinates and filter out boxes outside
    xml = os.path.join(TEST_XML_DIR, os.path.splitext(img_name)[0] + ".xml")
    region_coords = ET.parse(xml).getroot().find(".//{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}TextRegion/{http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15}Coords").attrib["points"]
    # print(f"[INFO] Filtering boxes outside main region for {img_name}")
    # print(f"[DEBUG] Region coords: {region_coords}")
    region_poly = []
    for xy in region_coords.split():
        x, y = map(int, xy.split(","))
        region_poly.append((x, y))
        
    region_poly_np = np.array(region_poly, dtype=np.int32).reshape(-1, 2)

    keep = []
    for box in boxes:
        x1, y1, x2, y2 = box.astype(int)
        cx = float((x1 + x2) // 2)
        cy = float((y1 + y2) // 2)

        val = cv2.pointPolygonTest(region_poly_np, (cx, cy), False)
        keep.append(val >= 0)

    boxes = boxes[np.array(keep)]

    root, region = create_page_xml(img_path, W, H)

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        crop = image[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        mask = segment_crop(crop)
        poly = mask_to_polygon(mask, x1, y1)
        if poly is None:
            continue

        coords = " ".join(f"{x},{y}" for x,y in poly)
        baseline = polygon_to_baseline(poly)
        baseline_pts = " ".join(f"{x},{y}" for x,y in baseline)

        tl = ET.SubElement(region, "TextLine", {
            "id": f"textline_{i}",
            "custom": "0"
        })
        ET.SubElement(tl, "Coords", {"points": coords})
        ET.SubElement(tl, "Baseline", {"points": baseline_pts})
        ET.SubElement(ET.SubElement(tl, "TextEquiv"), "Unicode").text = ""

    xml_str = minidom.parseString(
        ET.tostring(root, encoding="utf-8")
    ).toprettyxml(indent="  ")

    out_path = os.path.join(
        OUT_XML_DIR,
        os.path.splitext(img_name)[0] + ".xml"
    )

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(xml_str)

# ==================================================
print("[INFO] Done")

[INFO] Device: cuda
[INFO] Loading YOLO...
[INFO] Loading segmentation model...
[INFO] Models ready
[INFO] Found 10 images


Processing images: 100%|██████████| 10/10 [00:03<00:00,  3.20it/s]

[INFO] Done


In [10]:
import os
import subprocess
from tqdm import tqdm

def run_diva_evaluator(
    pred_xml_dir,
    out_csv_dir,
    gt_pixel_dir,
    gt_page_dir,
    img_dir,
    java_cp = "/usr/share/openjfx/lib/*:/home/artur/Thesis/DIVA_Line_Segmentation_Evaluator/out/artifacts/LineSegmentationEvaluator.jar",
    java_main = "ch.unifr.LineSegmentationEvaluatorTool"
):
    java_cp = os.path.expanduser(java_cp)
    gt_pixel_dir = os.path.expanduser(gt_pixel_dir)
    gt_page_dir = os.path.expanduser(gt_page_dir)
    img_dir = os.path.expanduser(img_dir)

    os.makedirs(out_csv_dir, exist_ok=True)

    pred_xmls = sorted(f for f in os.listdir(pred_xml_dir) if f.endswith(".xml"))

    for xml_name in tqdm(pred_xmls, desc="Running evaluator"):
        stem = os.path.splitext(xml_name)[0]

        pred_xml = os.path.join(pred_xml_dir, xml_name)
        gt_xml = os.path.join(gt_page_dir, f"{stem}.xml")
        gt_png = os.path.join(gt_pixel_dir, f"{stem}.png")
        overlap_img = os.path.join(img_dir, f"{stem}.jpg")
        out_csv = os.path.join(out_csv_dir, f"{stem}.csv")

        cmd = [
            "java",
            "-cp", java_cp,
            java_main,
            "-igt", gt_png,
            "-xgt", gt_xml,
            "-xp", pred_xml,
            "-overlap", overlap_img,
            "-csv",
        ]

        try:
            result = subprocess.run(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                check=True,
            )

            with open(out_csv, "w") as f:
                f.write(result.stdout)

        except subprocess.CalledProcessError as e:
            print(f"[ERROR] Evaluator failed on {stem}")
            print(e.stderr)

run_diva_evaluator(
    pred_xml_dir="./predictions_cb55/",
    out_csv_dir="./eval_csv_cb55/",
    gt_pixel_dir="~/Thesis/fsl-tl-manuscripts/00_data/DIVA-HisDB/CB55/pixel-level-gt-CB55/pixel-level-gt/public-test/",
    gt_page_dir="~/Thesis/fsl-tl-manuscripts/00_data/DIVA-HisDB/CB55/PAGE-gt-CB55-TASK-2/TASK-2/public-test/",
    img_dir="~/Thesis/fsl-tl-manuscripts/00_data/DIVA-HisDB/CB55/img-CB55/img/public-test/"
)

Running evaluator: 100%|██████████| 10/10 [02:52<00:00, 17.20s/it]


In [11]:
import pandas as pd

results_csv = pd.read_csv("./predictions_cb55/results.csv")
mean_line_iu = results_csv["LinesIU"].mean()
mean_pixel_iu = results_csv["PixelIU"].mean()
print(f"[RESULTS] Mean Line IU: {mean_line_iu:.2f}, Mean Pixel IU: {mean_pixel_iu:.2f}")

[RESULTS] Mean Line IU: 0.72, Mean Pixel IU: 0.74


In [12]:
import xml.etree.ElementTree as ET
import numpy as np
import cv2
import os

import pandas as pd
from typing import List, Tuple
from tqdm import tqdm


# ===================== CONFIG =====================
IOU_THRESH = 0.75
# =================================================


def parse_page_xml_polygons(xml_path: str) -> List[List[Tuple[int,int]]]:
    ns = {"ns": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"}
    tree = ET.parse(xml_path)
    root = tree.getroot()

    polygons = []
    for tl in root.findall(".//ns:TextLine", ns):
        coords = tl.find("ns:Coords", ns)
        if coords is None:
            continue
        pts = []
        for xy in coords.attrib["points"].split():
            x, y = map(int, xy.split(","))
            pts.append((x, y))
        if len(pts) >= 3:
            polygons.append(pts)

    return polygons


def polygon_to_mask(poly, H, W):
    mask = np.zeros((H, W), dtype=np.uint8)
    pts = np.array(poly, np.int32).reshape((-1, 1, 2))
    cv2.fillPoly(mask, [pts], 1)
    return mask


def match_score(mask_gt, mask_pred):
    inter = np.logical_and(mask_gt, mask_pred).sum()
    union = np.logical_or(mask_gt, mask_pred).sum()
    if union == 0:
        return 0.0
    return inter / union


def compute_DR_RA_FM(
    gt_xml: str,
    pred_xml: str,
    page_height: int,
    page_width: int,
    iou_thresh: float = IOU_THRESH
):
    gt_polys = parse_page_xml_polygons(gt_xml)
    pred_polys = parse_page_xml_polygons(pred_xml)

    N1 = len(gt_polys)     # GT lines
    N2 = len(pred_polys)  # Pred lines

    if N1 == 0 or N2 == 0:
        return {
            "DR": 0.0,
            "RA": 0.0,
            "FM": 0.0,
            "M": 0,
            "N_gt": N1,
            "N_pred": N2
        }

    gt_masks = [
        polygon_to_mask(p, page_height, page_width)
        for p in gt_polys
    ]
    pred_masks = [
        polygon_to_mask(p, page_height, page_width)
        for p in pred_polys
    ]

    used_gt = set()
    used_pred = set()
    M = 0

    # Greedy one-to-one matching
    for i, gt_mask in enumerate(gt_masks):
        best_j = -1
        best_score = 0.0

        for j, pr_mask in enumerate(pred_masks):
            if j in used_pred:
                continue
            score = match_score(gt_mask, pr_mask)
            if score > best_score:
                best_score = score
                best_j = j

        if best_score >= iou_thresh and best_j >= 0:
            M += 1
            used_gt.add(i)
            used_pred.add(best_j)

    DR = M / N1 if N1 > 0 else 0.0
    RA = M / N2 if N2 > 0 else 0.0
    FM = (2 * DR * RA) / (DR + RA) if (DR + RA) > 0 else 0.0

    return {
        "DR": round(DR, 2),
        "RA": round(RA, 2),
        "FM": round(FM, 2),
        "M": M,
        "N_gt": N1,
        "N_pred": N2
    }

def run_hscp_eval(
    pred_xml_dir: str,
    gt_xml_dir: str,
    test_img_dir: str
):
    img_files = sorted([
        f for f in os.listdir(test_img_dir)
        if f.lower().endswith((".jpg", ".png"))
    ])
    results = []
    for img_name in tqdm(img_files, desc="HSCP Evaluation"):
        stem = os.path.splitext(img_name)[0]
        img_path = os.path.join(test_img_dir, img_name)
        pred_xml = os.path.join(pred_xml_dir, f"{stem}.xml")
        gt_xml = os.path.join(gt_xml_dir, f"{stem}.xml")
        img = cv2.imread(img_path)
        H, W = img.shape[:2]
        metrics = compute_DR_RA_FM(
            gt_xml=gt_xml,
            pred_xml=pred_xml,
            page_height=H,
            page_width=W
        )
        metrics["image"] = img_name
        results.append(metrics)
    df = pd.DataFrame(results)
    
    out_csv = os.path.join(pred_xml_dir, "hscp_eval_results.csv")
    df.to_csv(out_csv, index=False)

    print(f"[INFO] HSCP evaluation results saved to {out_csv}")

run_hscp_eval(
    pred_xml_dir="./predictions_cb55/",
    gt_xml_dir="../00_data/DIVA-HisDB/CB55/PAGE-gt-CB55-TASK-2/TASK-2/public-test/",
    test_img_dir="../00_data/DIVA-HisDB/CB55/img-CB55/img/public-test/"
)
df = pd.read_csv("./predictions_cb55/hscp_eval_results.csv")

mean_DR = df["DR"].mean()
mean_RA = df["RA"].mean()
mean_FM = df["FM"].mean()
print(f"Mean DR: {mean_DR:.2f}, Mean RA: {mean_RA:.2f}, Mean FM: {mean_FM:.2f}")

HSCP Evaluation: 100%|██████████| 10/10 [01:16<00:00,  7.64s/it]

[INFO] HSCP evaluation results saved to ./predictions_cb55/hscp_eval_results.csv
Mean DR: 0.86, Mean RA: 0.85, Mean FM: 0.85
